# Does post-training change how a model quantizes?
Run top-to-bottom on a free Colab T4 (Runtime → Change runtime type → T4 GPU).
Four stages: **baseline eval → DPO training → re-eval → 4-bit quantization comparison**.
See `README.md` for the full runbook, the findings, and — importantly — the limits.

In [ ]:
!pip install -q transformers>=4.48 trl>=0.15 peft datasets accelerate bitsandbytes lm-eval sentencepiece
print('installs done — restart runtime if prompted, then continue')

## Stage 0 — config & helpers

In [ ]:
import json, torch
from pathlib import Path
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-0.5B'   # ungated base; fallback: 'TinyLlama/TinyLlama-1.1B'
OUT = Path('./dpo_project'); OUT.mkdir(exist_ok=True)
MMLU_TASKS = ['mmlu_high_school_mathematics','mmlu_elementary_mathematics','mmlu_college_computer_science']
MMLU_LIMIT = 30
PROMPTS = [
  'Explain why the sky is blue in one short paragraph.',
  'Write a polite email declining a meeting invitation.',
  'What are three tips for debugging a slow Python program?',
  'Summarize the plot of Romeo and Juliet in two sentences.',
  'Is it safe to share my password with tech support? Explain briefly.',
]

def get_tok(mid=MODEL_ID):
    t = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
    if t.pad_token is None: t.pad_token = t.eos_token
    return t

def save(obj, name):
    p = OUT / name
    p.write_text(json.dumps(obj, indent=2)); print('saved', p)

print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

## Stage 1 — baseline eval (two tracks)

**Track A — MMLU (multiple choice, scored by log-likelihood).** The lm-eval harness scores each question by comparing the model's log-probability over the A/B/C/D options — no free-form generation involved. This is the quantitative accuracy number (90 questions: 30 each across high-school math, elementary math, college CS).

**Track B — open-ended generations (qualitative).** A few free-form prompts where the model generates text freely. No score — you eyeball whether the base model rambles vs answers. This is the "vibe check" MMLU can't give you, and it's where DPO's effect is most visible.

In [ ]:
import lm_eval
res = lm_eval.simple_evaluate(model='hf',
    model_args=f'pretrained={MODEL_ID},dtype=float16,trust_remote_code=True',
    tasks=MMLU_TASKS, limit=MMLU_LIMIT, log_samples=True, batch_size=8)
base_mmlu = {t: res['results'][t].get('acc,none') for t in MMLU_TASKS}
print(base_mmlu); save(base_mmlu, 'baseline_mmlu.json')

In [ ]:
tok = get_tok()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True).eval()
base_gens = []
with torch.inference_mode():
    for p in PROMPTS:
        inp = tok(p, return_tensors='pt').to(model.device)
        g = model.generate(**inp, max_new_tokens=120, do_sample=False, pad_token_id=tok.eos_token_id)
        txt = tok.decode(g[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
        base_gens.append({'prompt': p, 'generation': txt})
        print(f'PROMPT: {p}\nGEN: {txt}\n' + '-'*60)
save(base_gens, 'baseline_generations.json')
del model; torch.cuda.empty_cache()

## Stage 2 — DPO fine-tuning (TRL + LoRA)
**Watch `rewards/margins`** in the output: the chosen-vs-rejected implicit-reward gap. Rising margin = the model is learning the preference.
`ref_model=None` means TRL builds a frozen reference from the base weights (with LoRA, the adapter is simply disabled for the reference) — no second model copy in VRAM.

In [ ]:
def as_text(msgs):
    return msgs.strip() if isinstance(msgs, str) else tok.apply_chat_template(msgs, tokenize=False).strip()

def split_prompt(conv):
    # dpo-mix-7k has no 'prompt' column: the prompt is the shared prefix
    # of the chosen/rejected conversations (everything but the last message)
    if isinstance(conv, str):
        return [], conv
    if isinstance(conv, list) and len(conv) > 1:
        return conv[:-1], conv[-1:]
    return [], conv

ds = load_dataset('argilla/dpo-mix-7k', split='train')
print('raw columns:', ds.column_names)
def fmt(b):
    prompts, chosens, rejecteds = [], [], []
    for c, r in zip(b['chosen'], b['rejected']):
        p_c, c_last = split_prompt(c)
        p_r, r_last = split_prompt(r)
        p = p_c if p_c else p_r
        prompts.append(as_text(p) if p else '')
        chosens.append(as_text(c_last))
        rejecteds.append(as_text(r_last))
    return {'prompt': prompts, 'chosen': chosens, 'rejected': rejecteds}
train_ds = ds.map(fmt, batched=True, remove_columns=ds.column_names)
print(len(train_ds), 'preference pairs | columns:', train_ds.column_names)
print('example prompt:', train_ds[0]['prompt'][:200])
print('example chosen:', train_ds[0]['chosen'][:200])

In [ ]:
from peft import LoraConfig, TaskType
from trl import DPOConfig, DPOTrainer

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
args = DPOConfig(output_dir=str(OUT/'dpo_adapter'), beta=0.1,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,   # effective batch 8
    max_steps=500, max_length=512,
    learning_rate=5e-5, lr_scheduler_type='cosine', warmup_steps=50,
    logging_steps=10, save_steps=250, fp16=True,                    # T4: no bf16
    gradient_checkpointing=True, remove_unused_columns=False, report_to='none')
trainer = DPOTrainer(model=model, ref_model=None, args=args, train_dataset=train_ds,
                     processing_class=tok,
                     peft_config=LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
                         lora_dropout=0.05, target_modules=['q_proj','k_proj','v_proj','o_proj']))
trainer.train()
trainer.save_model(str(OUT/'dpo_adapter')); tok.save_pretrained(str(OUT/'dpo_adapter'))
save(trainer.state.log_history, 'dpo_train_log.json')
print('adapter saved')

In [ ]:
# Plot the reward margin curve — the single most important diagnostic
import matplotlib.pyplot as plt
log = json.loads((OUT/'dpo_train_log.json').read_text())
steps=[e['step'] for e in log if 'rewards/margins' in e]
marg=[e['rewards/margins'] for e in log if 'rewards/margins' in e]
plt.plot(steps, marg); plt.xlabel('step'); plt.ylabel('rewards/margins')
plt.title('DPO reward margin (rising = learning the preference)'); plt.show()

In [ ]:
# Merge LoRA into base weights -> a plain HF checkpoint for eval
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='cpu', trust_remote_code=True)
merged = PeftModel.from_pretrained(base, str(OUT/'dpo_adapter')).merge_and_unload()
merged.save_pretrained(str(OUT/'dpo_merged')); tok.save_pretrained(str(OUT/'dpo_merged'))
print('merged checkpoint saved')
del base, merged; torch.cuda.empty_cache()

## Stage 3 — re-eval the DPO checkpoint
Same two tracks as stage 1. **Track A:** MMLU may dip slightly — that's the **alignment tax** (capability regression from preference tuning). Note the n=30 caveat: each question is worth 0.033, so only large deltas are signal. **Track B:** generations should now *answer helpfully* instead of rambling — that's DPO working.

In [ ]:
MPATH = str(OUT/'dpo_merged')
res = lm_eval.simple_evaluate(model='hf',
    model_args=f'pretrained={MPATH},dtype=float16,trust_remote_code=True',
    tasks=MMLU_TASKS, limit=MMLU_LIMIT, batch_size=8)
dpo_mmlu = {t: res['results'][t].get('acc,none') for t in MMLU_TASKS}
save(dpo_mmlu, 'dpo_mmlu.json')
print(f"{'task':40s} {'base':>6s} {'dpo':>6s} {'delta':>7s}")
for t in MMLU_TASKS:
    b, d = base_mmlu[t] or 0, dpo_mmlu[t] or 0
    print(f'{t:40s} {b:6.3f} {d:6.3f} {d-b:+7.3f}')

In [ ]:
tok2 = get_tok(MPATH)
m2 = AutoModelForCausalLM.from_pretrained(MPATH, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True).eval()
dpo_gens = []
with torch.inference_mode():
    for p in PROMPTS:
        inp = tok2(p, return_tensors='pt').to(m2.device)
        g = m2.generate(**inp, max_new_tokens=120, do_sample=False, pad_token_id=tok2.eos_token_id)
        txt = tok2.decode(g[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
        dpo_gens.append({'prompt': p, 'generation': txt})
save(dpo_gens, 'dpo_generations.json')
for b, d in zip(base_gens, dpo_gens):
    print(f"PROMPT: {b['prompt']}\nBASE: {b['generation'][:220]}\nDPO:  {d['generation'][:220]}\n"+'='*70)
del m2; torch.cuda.empty_cache()

## Stage 4 — 4-bit quantization sensitivity: base vs aligned
The key question: does the **aligned** checkpoint degrade differently under 4-bit quantization than the base? We load each checkpoint in fp16 and in 4-bit NF4 (bitsandbytes, weight-only) and compare perplexity plus an in-process multiple-choice MMLU scorer (log-likelihood over A/B/C/D, same idea as lm-eval). If DPO introduced a new quantization fragility mode, the aligned model's degradation would be larger — a null result (both degrade alike) is itself the finding.

In [ ]:
# ---- Stage 4: fp16 vs 4-bit NF4, base vs DPO checkpoint ----
from tqdm import tqdm
from datasets import load_dataset as _lds
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
SUBJECTS = [t.replace('mmlu_', '') for t in MMLU_TASKS]  # -> cais/mmlu config names

def ppl_corpus(n_chars=200_000):
    try:
        wt = _lds('wikitext', 'wikitext-2-raw-v1', split='test')
        return '\n\n'.join(wt['text'])[:n_chars]
    except Exception as e:
        print(f'wikitext load failed ({type(e).__name__}); falling back to dpo-mix-7k test text')
        ds = _lds('argilla/dpo-mix-7k', split='test')
        return '\n\n'.join(as_text(c) for c in ds['chosen'][:400])[:n_chars]

@torch.inference_mode()
def perplexity(model, tok, stride=512):
    ids = tok(ppl_corpus(), return_tensors='pt')['input_ids'].to(model.device)
    nlls = []
    for i in tqdm(range(0, ids.size(1) - stride, stride), leave=False):
        c = ids[:, i:i+stride]
        l = model(c).logits
        nlls.append(torch.nn.functional.cross_entropy(l[0, :-1].reshape(-1, l.size(-1)), c[0, 1:].reshape(-1)))
    return torch.exp(torch.stack(nlls).mean()).item()

@torch.inference_mode()
def mmlu_manual(model, tok, limit=30):
    """Multiple-choice MMLU scored by log-likelihood over A/B/C/D (same idea as lm-eval)."""
    acc = {}
    for subj in SUBJECTS:
        ds = _lds('cais/mmlu', subj, split='test').select(range(limit))
        correct = 0
        for ex in tqdm(ds, desc=f'mmlu/{subj}', leave=False):
            prompt = 'Question: ' + ex['question'] + '\n' + '\n'.join(
                f'{chr(65+i)}. {ch}' for i, ch in enumerate(ex['choices'])) + '\nAnswer:'
            inp = tok(prompt, return_tensors='pt').to(model.device)
            L = inp['input_ids'].size(1)
            scores = []
            for letter in 'ABCD':
                cont = tok(' ' + letter, return_tensors='pt')['input_ids'].to(model.device)
                full = torch.cat([inp['input_ids'], cont], dim=1)
                logits = model(full).logits[0]
                lp = torch.log_softmax(logits[L-1:-1], dim=-1)
                scores.append(lp.gather(-1, cont[0].unsqueeze(-1)).sum().item())
            correct += int(torch.tensor(scores).argmax()) == ex['answer']
        acc[subj] = correct / len(ds)
    return acc

rep = {}
for name, path in [('base', MODEL_ID), ('dpo', str(OUT / 'dpo_merged'))]:
    tok = get_tok(path)
    rep[name] = {}
    for fmt, quant in [('fp16', False), ('nf4', True)]:
        if quant:
            m = AutoModelForCausalLM.from_pretrained(path, quantization_config=bnb,
                                                    device_map='auto', trust_remote_code=True).eval()
        else:
            m = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16,
                                                    device_map='auto', trust_remote_code=True).eval()
        ppl = perplexity(m, tok)
        acc = mmlu_manual(m, tok)
        rep[name][fmt] = {'ppl': ppl, 'mmlu': acc}
        print(f"{name}/{fmt}: ppl={ppl:.2f} mmlu={ {k: round(v,3) for k,v in acc.items()} }")
        del m; torch.cuda.empty_cache()

print('\ndegradation vs fp16:')
for name in rep:
    dp = 100 * (rep[name]['nf4']['ppl'] - rep[name]['fp16']['ppl']) / rep[name]['fp16']['ppl']
    dm = {k: round(rep[name]['nf4']['mmlu'][k] - rep[name]['fp16']['mmlu'][k], 3) for k in SUBJECTS}
    print(f'{name}/nf4: ppl +{dp:.1f}%   mmlu delta {dm}')
save(rep, 'nf4_compare.json')

## Wrap-up — what this demonstrates
- Ran DPO end-to-end on a 0.5B model with TRL: watched the reward margin rise and the policy drift from its frozen reference (KL budget β=0.1).
- Measured the **alignment tax** on MMLU — honestly: n=30/subject is noise-level, so the claim is "no collapse," not a precise tax.
- Characterized **4-bit quantization sensitivity of aligned vs base checkpoints** — and found a null result: the aligned model degrades like the base.

Limits, stated plainly: 0.5B scale, 500 DPO steps, one dataset, one seed, weight-only NF4. A study, not research.

Next steps to deepen it: try β=0.05 vs 0.2 and compare drift; swap DPO for GRPO on a verifiable task (TRL `GRPOTrainer`); run the same pipeline on a 1–3B model.